# Task F — оценка стоимости автомобиля

Регрессия цены автомобиля по табличным параметрам и изображениям.


## Зафиксированный результат

**Leaderboard score: 0.820.** Validation median APE: **0.2082** после постобработки.

> Метрика перенесена из авторского экспериментального ноутбука. Тяжёлые логи обучения удалены, чтобы решение хорошо отображалось на GitHub.


## Запуск

Положите данные в каталог `data/`, установите зависимости из корневого `requirements.txt` и последовательно выполните ячейки. Артефакты и сабмит будут записаны в `outputs/`.


In [ ]:
import os
from pathlib import Path
import torch
import timm
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

# ============================================================
# НАСТРОЙКИ (поменяй пути под себя)
# ============================================================
DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_IMAGES_DIR = DATA_DIR / "train_images"
TEST_IMAGES_DIR = DATA_DIR / "test_images"
DEVICE = "mps"

# Если у тебя Mac с чипом M1/M2/M3, можно попробовать:
# DEVICE = torch.device("mps")

print(f"Устройство: {DEVICE}")

# ============================================================
# ЗАГРУЗКА МОДЕЛИ
# ============================================================
print("Загрузка модели SigLIP...")

# Создаём модель. Это ViT (Vision Transformer), обученный на огромном датасете.
# Он превращает картинку 224x224 в вектор из 768 чисел.
# num_classes=0 — убираем классификатор, оставляем только эмбеддинг.
img_model = timm.create_model(
    'vit_base_patch16_clip_224.laion2b_ft_in12k',
    pretrained=True,
    num_classes=0
)
img_model = img_model.to(DEVICE)
img_model.eval()  # Режим инференса (не обучаем)

# Создаём трансформацию: картинка -> тензор [3, 224, 224]
# Модель сама знает, какой размер и нормализация ей нужны
data_config = timm.data.resolve_model_data_config(img_model)
img_transform = timm.data.create_transform(**data_config, is_training=False)

print(f"Модель загружена. Размер эмбеддинга: {img_model.num_features}")
# img_model.num_features покажет размерность вектора (обычно 768)


# ============================================================
# ФУНКЦИЯ: ИЗВЛЕЧЬ ЭМБЕДДИНГИ ИЗ ОДНОГО ОБЪЕКТА
# ============================================================
def get_object_embedding(obj_id, img_dir):
    """
    Берёт все фото одного объекта (до 4 штук),
    прогоняет каждое через модель,
    усредняет эмбеддинги.
    
    Возвращает один вектор (эмбеддинг объекта).
    """
    embeddings = []
    
    # Пробуем загрузить фото: ID_0.jpg, ID_1.jpg, ID_2.jpg, ID_3.jpg
    for img_idx in range(4):
        img_path = os.path.join(img_dir, f"{obj_id}_{img_idx}.jpg")
        
        # Если файла нет — пропускаем
        if not os.path.exists(img_path):
            continue
        
        try:
            # Открываем картинку и конвертируем в RGB
            img = Image.open(img_path).convert("RGB")
            
            # Применяем трансформацию (resize, crop, normalize)
            img_tensor = img_transform(img)
            
            # Добавляем размерность batch: [3,224,224] -> [1,3,224,224]
            img_tensor = img_tensor.unsqueeze(0).to(DEVICE)
            
            # Прогоняем через модель без вычисления градиентов
            with torch.no_grad():
                emb = img_model(img_tensor)  # результат: [1, 768]
            
            # Переносим на CPU и превращаем в numpy
            emb_np = emb.cpu().numpy().flatten()  # [768]
            embeddings.append(emb_np)
            
        except Exception as e:
            # Если картинка битая — пропускаем
            continue
    
    # Если не удалось загрузить ни одного фото — возвращаем нули
    if len(embeddings) == 0:
        return np.zeros(img_model.num_features)
    
    # Усредняем все эмбеддинги
    mean_embedding = np.mean(embeddings, axis=0)
    
    return mean_embedding


# ============================================================
# ФУНКЦИЯ: ИЗВЛЕЧЬ ЭМБЕДДИНГИ ДЛЯ ВСЕХ ОБЪЕКТОВ
# ============================================================
def get_all_embeddings(ids, img_dir, desc="Processing"):
    """
    Проходит по всем ID и извлекает эмбеддинги.
    Возвращает матрицу [количество_объектов, 768].
    """
    embeddings = []
    
    for obj_id in tqdm(ids, desc=desc):
        emb = get_object_embedding(obj_id, img_dir)
        embeddings.append(emb)
    
    return np.array(embeddings)


# ============================================================
# ЗАПУСК ИЗВЛЕЧЕНИЯ
# ============================================================

# Загружаем ID из таблиц
train_df = pd.read_parquet(DATA_DIR / "train_dataset.parquet")
test_df = pd.read_parquet(DATA_DIR / "test_dataset.parquet")

train_ids = train_df['ID'].values
test_ids = test_df['ID'].values

print(f"\nИзвлечение эмбеддингов для {len(train_ids)} объектов трейна...")
train_img_emb = get_all_embeddings(train_ids, TRAIN_IMAGES_DIR, desc="Train")

print(f"\nИзвлечение эмбеддингов для {len(test_ids)} объектов теста...")
test_img_emb = get_all_embeddings(test_ids, TEST_IMAGES_DIR, desc="Test")

print(f"\nTrain эмбеддинги: {train_img_emb.shape}")
print(f"Test эмбеддинги:  {test_img_emb.shape}")

# Сохраняем, чтобы не пересчитывать
np.save(OUTPUT_DIR / "train_img_emb.npy", train_img_emb)
np.save(OUTPUT_DIR / "test_img_emb.npy", test_img_emb)

print("Эмбеддинги сохранены!")

# Обучение модели и формирование сабмита
from pathlib import Path

import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split

DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# ШАГ 1: ЗАГРУЗКА ДАННЫХ
# ============================================================
print("=" * 50)
print("ШАГ 1: Загрузка данных")
print("=" * 50)

train_df = pd.read_parquet(DATA_DIR / "train_dataset.parquet")
test_df = pd.read_parquet(DATA_DIR / "test_dataset.parquet")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

print(f"Размер трейна: {train_df.shape}")
print(f"Размер теста:  {test_df.shape}")


# ============================================================
# ШАГ 2: ОБРАБОТКА ПРОПУСКОВ В КАТЕГОРИЯХ
# ============================================================
print("\n" + "=" * 50)
print("ШАГ 2: Обработка пропусков в категориях")
print("=" * 50)

# Список категориальных колонок
cat_cols = [
    'body_type', 'drive_type', 'engine_type', 'color',
    'pts', 'steering_wheel', 'equipment',
    'audiosistema', 'diski', 'electropodemniki',
    'fary', 'salon', 'upravlenie_klimatom', 'usilitel_rul',
    'owners_count', 'crashes_count'
]

# Для каждой категориальной колонки:
# - если значение пустое (NaN), заменяем на строку 'missing'
# - приводим всё к строковому типу
for col in cat_cols:
    train_df[col] = train_df[col].fillna('missing').astype(str)
    test_df[col] = test_df[col].fillna('missing').astype(str)

print("Пропуски в категориях заполнены словом 'missing'")


# ============================================================
# ШАГ 3: ОБРАБОТКА МУЛЬТИ-ОПЦИЙ
# ============================================================
print("\n" + "=" * 50)
print("ШАГ 3: Обработка мульти-опций")
print("=" * 50)

# Находим все колонки, название которых заканчивается на '_mult'
mult_cols = []
for col in train_df.columns:
    if col.endswith('_mult'):
        mult_cols.append(col)

print(f"Найдено мульти-колонок: {len(mult_cols)}")
print(f"Их названия: {mult_cols}")


# Функция: считает количество реальных опций в списке
# Например: ['Обогрев сидений', 'Подогрев руля', None] -> ответ 2
def count_options(lst):
    # Если значение пустое или не итерируемое — возвращаем 0
    if lst is None:
        return 0

    # Пробуем пройтись по элементам
    try:
        count = 0
        for item in lst:
            # Пропускаем None и строку 'None'
            if item is None:
                continue
            if str(item).lower() == 'none':
                continue
            # Если дошли сюда — это реальная опция
            count += 1
        return count
    except TypeError:
        # Если lst не список (например, число или строка) — возвращаем 0
        return 0


# Применяем функцию к каждой мульти-колонке
for col in mult_cols:
    new_col_name = col + '_count'

    train_df[new_col_name] = train_df[col].apply(count_options)
    test_df[new_col_name] = test_df[col].apply(count_options)

    print(f"  {col} -> {new_col_name} (среднее: {train_df[new_col_name].mean():.2f})")

# Создаём колонку с общим количеством всех опций
# Берём все колонки типа '..._count' и суммируем по строкам
count_col_names = [col + '_count' for col in mult_cols]

train_df['total_options'] = train_df[count_col_names].sum(axis=1)
test_df['total_options'] = test_df[count_col_names].sum(axis=1)

print(f"\nОбщее число новых признаков: {len(mult_cols) + 1}")
print(f"Среднее кол-во опций в трейне: {train_df['total_options'].mean():.2f}")


# ============================================================
# ШАГ 4: СОЗДАНИЕ ВЗАИМОДЕЙСТВИЙ
# ============================================================
print("\n" + "=" * 50)
print("ШАГ 4: Создание взаимодействий")
print("=" * 50)

# Сначала создаём числовые копии owners_count и crashes_count
# потому что сейчас они строки ('1', '2', '3', 'missing')
for df in [train_df, test_df]:
    # Пытаемся превратить строку в число. Если не получается — NaN
    df['owners_num'] = pd.to_numeric(df['owners_count'], errors='coerce')
    df['crashes_num'] = pd.to_numeric(df['crashes_count'], errors='coerce')

# Теперь создаём признаки-взаимодействия
for df in [train_df, test_df]:
    # 1. Пробег на одного владельца
    #    Если пробег 100000 и 2 владельца -> 50000 на владельца
    #    +1 чтобы не делить на ноль
    df['mileage_per_owner'] = df['mileage'] / (df['owners_num'] + 1)

    # 2. Логарифм пробега (чтобы модель лучше видела разницу
    #    между 10000 и 50000, а не между 200000 и 250000)
    df['mileage_log'] = np.log1p(df['mileage'])

    # 3. Количество ДТП на одного владельца
    df['crashes_per_owner'] = df['crashes_num'] / (df['owners_num'] + 1)

print("Созданы признаки: mileage_per_owner, mileage_log, crashes_per_owner")
print(f"Пример:\n{train_df[['mileage', 'owners_num', 'mileage_per_owner']].head(3)}")

# ============================================================
# ТАРГЕТ-ЭНКОДИНГ
# ============================================================
print("Создание таргет-энкодинга...")

from sklearn.model_selection import KFold

# Категории, для которых делаем энкодинг
te_cols = [
    'body_type', 'engine_type', 'drive_type', 'color',
    'equipment', 'pts', 'steering_wheel',
    'owners_count', 'crashes_count'
]

# Количество фолдов для энкодинга
n_folds = 5
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
train_df['log_price'] = np.log1p(train_df['price_TARGET'])
# Глобальное среднее лог-цены (для заполнения пропусков)
global_mean = train_df['log_price'].mean()

for col in te_cols:
    # --- ТРЕЙН: считаем через фолды (чтобы не было утечки) ---
    train_df[col + '_te'] = global_mean  # начальное значение

    for train_idx, val_idx in kf.split(train_df):
        # На обучающем фолде считаем среднюю цену для каждой категории
        fold_means = train_df.iloc[train_idx].groupby(col)['log_price'].mean()

        # Применяем эти средние к валидационному фолду
        mapped = train_df.iloc[val_idx][col].map(fold_means)

        # Если категория не встречалась в фолде — ставим глобальное среднее
        train_df.iloc[val_idx, train_df.columns.get_loc(col + '_te')] = \
            mapped.fillna(global_mean).values

    # --- ТЕСТ: считаем средние на ВЕСЬ трейн ---
    full_means = train_df.groupby(col)['log_price'].mean()
    test_df[col + '_te'] = test_df[col].map(full_means).fillna(global_mean)

    print(f"  {col}_te: {train_df[col + '_te'].nunique():.0f} уникальных значений")

# Добавляем в числовые признаки
te_feature_names = [col + '_te' for col in te_cols]
# num_features += te_feature_names  # добавим позже при формировании списка
print(f"Добавлено {len(te_feature_names)} таргет-энкодинг признаков")


# ============================================================
# ШАГ 4.5: ПОДКЛЮЧЕНИЕ ЭМБЕДДИНГОВ КАРТИНОК
# ============================================================
print("\n" + "=" * 50)
print("ШАГ 4.5: Подключение эмбеддингов картинок")
print("=" * 50)

from sklearn.decomposition import PCA

# Загружаем сохранённые эмбеддинги
train_img_emb = np.load(OUTPUT_DIR / "train_img_emb.npy")
test_img_emb = np.load(OUTPUT_DIR / "test_img_emb.npy")

print(f"Train эмбеддинги: {train_img_emb.shape}")  # (70000, 768)
print(f"Test эмбеддинги:  {test_img_emb.shape}")   # (25000, 768)

# Сжимаем 768 признаков -> 50 с помощью PCA
# Это ускоряет обучение в 10+ раз и убирает шум
N_PCA = 150
pca = PCA(n_components=N_PCA, random_state=42)

train_img_pca = pca.fit_transform(train_img_emb)
test_img_pca = pca.transform(test_img_emb)  # ← ВАЖНО: transform, а не fit_transform!

print(f"PCA сжал: 768 -> {N_PCA} признаков")
print(f"Объяснённая дисперсия: {pca.explained_variance_ratio_.sum():.2%}")

# Создаём названия колонок
img_feature_names = [f"img_{i}" for i in range(N_PCA)]

# Присваиваем ПРАВИЛЬНЫЕ массивы каждому датафрейму
for i, name in enumerate(img_feature_names):
    train_df[name] = train_img_pca[:, i]   # ← трейну трейновые
    test_df[name] = test_img_pca[:, i]     # ← тесту тестовые (ИСПРАВЛЕНО!)

print(f"Добавлено {N_PCA} признаков картинок")
# ============================================================
# ШАГ 6.6: РАСШИРЕННЫЕ KNN-ПРИЗНАКИ
# ============================================================
print("\n" + "=" * 50)
print("ШАГ 6.6: Расширенные KNN-признаки")
print("=" * 50)

from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import KFold as KFold2

K = 50  # увеличили соседей

knn_features = [
    'knn_median', 'knn_mean', 'knn_std',
    'knn_q25', 'knn_q75', 'knn_q10', 'knn_q90',
    'knn_min', 'knn_max', 'knn_range'
]

train_knn = {name: np.zeros(len(train_df)) for name in knn_features}
test_knn = {name: np.zeros(len(test_df)) for name in knn_features}

log_prices = train_df['log_price'].values
kf_knn = KFold2(n_splits=5, shuffle=True, random_state=42)

# Трейн через фолды
print("KNN для трейна (через фолды)...")
for train_idx, val_idx in kf_knn.split(train_img_emb):
    knn_fold = NearestNeighbors(n_neighbors=K, metric='cosine', n_jobs=-1)
    knn_fold.fit(train_img_emb[train_idx])
    distances, indices = knn_fold.kneighbors(train_img_emb[val_idx])
    
    for i, vi in enumerate(val_idx):
        np_prices = log_prices[train_idx[indices[i]]]
        train_knn['knn_median'][vi] = np.median(np_prices)
        train_knn['knn_mean'][vi] = np.mean(np_prices)
        train_knn['knn_std'][vi] = np.std(np_prices)
        train_knn['knn_q25'][vi] = np.percentile(np_prices, 25)
        train_knn['knn_q75'][vi] = np.percentile(np_prices, 75)
        train_knn['knn_q10'][vi] = np.percentile(np_prices, 10)
        train_knn['knn_q90'][vi] = np.percentile(np_prices, 90)
        train_knn['knn_min'][vi] = np.min(np_prices)
        train_knn['knn_max'][vi] = np.max(np_prices)
        train_knn['knn_range'][vi] = np.max(np_prices) - np.min(np_prices)

# Тест
print("KNN для теста...")
knn_full = NearestNeighbors(n_neighbors=K, metric='cosine', n_jobs=-1)
knn_full.fit(train_img_emb)
test_distances, test_indices = knn_full.kneighbors(test_img_emb)

for i in range(len(test_df)):
    np_prices = log_prices[test_indices[i]]
    test_knn['knn_median'][i] = np.median(np_prices)
    test_knn['knn_mean'][i] = np.mean(np_prices)
    test_knn['knn_std'][i] = np.std(np_prices)
    test_knn['knn_q25'][i] = np.percentile(np_prices, 25)
    test_knn['knn_q75'][i] = np.percentile(np_prices, 75)
    test_knn['knn_q10'][i] = np.percentile(np_prices, 10)
    test_knn['knn_q90'][i] = np.percentile(np_prices, 90)
    test_knn['knn_min'][i] = np.min(np_prices)
    test_knn['knn_max'][i] = np.max(np_prices)
    test_knn['knn_range'][i] = np.max(np_prices) - np.min(np_prices)

# Добавляем в датафреймы
for name in knn_features:
    train_df[name] = train_knn[name]
    test_df[name] = test_knn[name]

print(f"Добавлено {len(knn_features)} KNN-признаков")

# ============================================================
# ШАГ 5: ФОРМИРОВАНИЕ СПИСКОВ ПРИЗНАКОВ
# ============================================================
print("\n" + "=" * 50)
print("ШАГ 5: Формирование списков признаков")
print("=" * 50)

# Категориальные признаки (строки, которые CatBoost закодирует сам)
cat_features = [
    'body_type', 'drive_type', 'engine_type', 'color',
    'pts', 'steering_wheel', 'equipment',
    'audiosistema', 'diski', 'electropodemniki',
    'fary', 'salon', 'upravlenie_klimatom', 'usilitel_rul',
    'owners_count', 'crashes_count'
]

num_features = [
    'mileage', 'latitude', 'longitude', 'doors_number',
    'total_options', 'mileage_per_owner', 'mileage_log', 'crashes_per_owner'
]

# Мульти-опции
for col in mult_cols:
    num_features.append(col + '_count')

# Таргет-энкодинг
num_features += te_feature_names

# Картинки (НОВОЕ)
num_features += img_feature_names

num_features += knn_features
# Всё вместе
all_features = cat_features + num_features


# ============================================================
# ШАГ 6: ЦЕЛЕВАЯ ПЕРЕМЕННАЯ
# ============================================================
print("\n" + "=" * 50)
print("ШАГ 6: Целевая переменная")
print("=" * 50)

# Логарифмируем цену, потому что:
# - цены имеют длинный хвост (много дешёвых, мало дорогих)
# - логарифм делает распределение более симметричным
# - ошибки становятся относительными (что важно для medianAPE)
train_df['log_price'] = np.log1p(train_df['price_TARGET'])

print(f"Мин цена:  {train_df['price_TARGET'].min():,.0f} руб")
print(f"Макс цена: {train_df['price_TARGET'].max():,.0f} руб")
print(f"Медиана:   {train_df['price_TARGET'].median():,.0f} руб")
print(f"Лог-медиана: {train_df['log_price'].median():.4f}")


# ============================================================
# ШАГ 7: РАЗДЕЛЕНИЕ НА ТРЕЙН И ВАЛИДАЦИЮ
# ============================================================
print("\n" + "=" * 50)
print("ШАГ 7: Разделение данных")
print("=" * 50)

# 80% на обучение, 20% на проверку
train_split, val_split = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42
)

# Признаки и таргет для обучения
X_train = train_split[all_features]
y_train = train_split['log_price']

# Признаки и таргет для валидации
X_val = val_split[all_features]
y_val = val_split['log_price']

print(f"Обучающая выборка:   {X_train.shape[0]} объектов")
print(f"Валидационная выборка: {X_val.shape[0]} объектов")


# ============================================================
# ШАГ 8: ОБУЧЕНИЕ МОДЕЛИ
# ============================================================
print("\n" + "=" * 50)
print("ШАГ 8: Обучение CatBoost")
print("=" * 50)

model = CatBoostRegressor(
    iterations=2000,          # Максимум 2000 деревьев
    learning_rate=0.05,       # Маленький шаг = точнее, но медленнее
    depth=8,                  # Глубина каждого дерева
    loss_function='MAE',      # Минимизируем абсолютную ошибку в логах
    eval_metric='MAE',        # Метрика для ранней остановки
    random_seed=42,           # Для воспроизводимости
    early_stopping_rounds=200, # Если 200 итераций нет улучшения — стоп
    verbose=100,              # Печатать прогресс каждые 100 итераций
    l2_leaf_reg=3,            # Регуляризация (чтобы не переобучиться)
)

model.fit(
    X_train, y_train,
    cat_features=cat_features,   # Говорим какие колонки категориальные
    eval_set=(X_val, y_val),     # На этом наборе следим за переобучением
    use_best_model=True          # Используем лучшую итерацию, а не последнюю
)

print(f"\nЛучшая итерация: {model.get_best_iteration()}")


# ============================================================
# ШАГ 9: ОЦЕНКА НА ВАЛИДАЦИИ
# ============================================================
print("\n" + "=" * 50)
print("ШАГ 9: Оценка модели")
print("=" * 50)

# Предсказываем в лог-пространстве
val_preds_log = model.predict(X_val)

# Преобразуем обратно в рубли
val_preds = np.expm1(val_preds_log)
val_true = np.expm1(y_val)

# Функция метрики: медиана абсолютной процентной ошибки
def median_ape(y_true, y_pred):
    # Защита от деления на ноль
    y_true = np.maximum(y_true, 1)
    # Считаем процентную ошибку для каждого объекта
    errors = np.abs(y_pred - y_true) / y_true
    # Берём медиану
    return np.median(errors)

score = median_ape(val_true, val_preds)
print(f"\n✓ Validation medianAPE: {score:.4f}")
print(f"  Это значит: медианная ошибка = {score*100:.1f}%")
print(f"  (меньше = лучше)")

# Смотрим на примеры ошибок
val_results = val_split[['ID', 'body_type', 'mileage', 'price_TARGET']].copy()
val_results['pred_price'] = val_preds
val_results['error_pct'] = np.abs(val_preds - val_true) / val_true * 100

print("\n--- 5 худших предсказаний ---")
worst = val_results.sort_values('error_pct', ascending=False).head(5)
for _, row in worst.iterrows():
    print(f"  ID={row['ID']}: истинная={row['price_TARGET']:,.0f}, "
          f"предсказание={row['pred_price']:,.0f}, ошибка={row['error_pct']:.0f}%")

print("\n--- 5 лучших предсказаний ---")
best = val_results.sort_values('error_pct').head(5)
for _, row in best.iterrows():
    print(f"  ID={row['ID']}: истинная={row['price_TARGET']:,.0f}, "
          f"предсказание={row['pred_price']:,.0f}, ошибка={row['error_pct']:.0f}%")


# ============================================================
# ШАГ 10: ПРЕДСКАЗАНИЕ НА ТЕСТЕ
# ============================================================
print("\n" + "=" * 50)
print("ШАГ 10: Предсказание на тесте")
print("=" * 50)

# Берём только те признаки, на которых обучали
X_test = test_df[all_features]

# Предсказываем в лог-пространстве
test_preds_log = model.predict(X_test)
test_preds = np.expm1(test_preds_log)
# ============================================================
# ШАГ 10.5: ПОСТ-ОБРАБОТКА
# ============================================================
print("\nПодбор множителя...")

best_delta = 0
best_score = score

for delta in np.linspace(-0.15, 0.15, 301):
    adjusted = val_preds * np.exp(delta)
    adjusted_score = median_ape(val_true, adjusted)
    if adjusted_score < best_score:
        best_score = adjusted_score
        best_delta = delta

print(f"Сдвиг: {best_delta:.4f}")
print(f"Score до: {score:.4f}, после: {best_score:.4f}")

# Применяем к тесту
test_preds = test_preds * np.exp(best_delta)

# Защита: цена не может быть отрицательной или нулевой
test_preds = np.clip(test_preds, 1, None)

print(f"Мин предсказание: {test_preds.min():,.0f} руб")
print(f"Макс предсказание: {test_preds.max():,.0f} руб")
print(f"Медиана предсказаний: {np.median(test_preds):,.0f} руб")


# ============================================================
# ШАГ 11: СОХРАНЕНИЕ РЕЗУЛЬТАТА
# ============================================================
print("\n" + "=" * 50)
print("ШАГ 11: Сохранение результата")
print("=" * 50)

submission = pd.DataFrame({
    'ID': test_df['ID'],
    'target': test_preds
})

output_path = OUTPUT_DIR / "submission.csv"
submission.to_csv(output_path, index=False)

print(f"✓ Файл сохранён: {output_path}")
print(f"  Количество строк: {len(submission)}")
print(f"  Первые 5 строк:")
print(submission.head())
print("\nГотово! Можно отправлять на лидерборд.")
